# Crear un usuario

Guarda un usuario nuevo con su nombre y su email.

## La petición

`POST /users`, con este cuerpo:

```json
{ "name": "Ada", "email": "ada@example.com" }
```

## Qué pasa en la base

Añade una fila. La base pone el id y las dos fechas. El email se guarda en minúsculas.

## Las respuestas

| Estado | Qué se ve |
| --- | --- |
| 201 | El usuario creado |
| 400 | `Name is empty` o `Email is invalid` |
| 409 | `Email is already used`. Pasa si lo ejecutas dos veces sin borrar a Ada |
| 500 | `Could not create the user`. No se ha llegado a la base |

## Qué tiene que estar en marcha

Antes de ejecutarlo, lanza `pnpm start` y `pnpm node-http`, cada uno en su terminal.


In [ ]:
# Lanza programas y lee las variables del sistema.
import os
import subprocess
# Permite subir de carpeta hasta encontrar la app.
from pathlib import Path

# Busca la carpeta de la app, para que Node encuentre sus paquetes.
app = Path.cwd()
while app != app.parent and not (app / "src" / "logic").is_dir():
    app = app.parent
if not (app / "src" / "logic").is_dir():
    raise SystemExit("Cannot find the app.")

# El programa de Node que llama a la API.
code = r"""
// Pinta tablas en la salida del cuaderno.
import Table from "cli-table3";

// Dirección de la API.
const ORIGIN = "http://127.0.0.1:3004";

// Datos del usuario. Puedes cambiarlos.
const name = "Ada";
const email = "ada@example.com";

// Lanza la petición.
const response = await fetch(`${ORIGIN}/users`, {
  method: "POST",
  headers: { "content-type": "application/json" },
  body: JSON.stringify({ name, email }),
});

// Muestra el método, la ruta y el estado de la respuesta.
console.log(`POST /users: ${response.status}`);

// Lee la respuesta. Algunas respuestas no traen nada.
const text = await response.text();
const result = text === "" ? null : JSON.parse(text);

// Si ha fallado, se ve el mensaje y no la tabla.
if (!response.ok) {
  console.log(result?.message ?? "");
  process.exit(0);
}

// Prepara la tabla con los datos de cada usuario.
const table = new Table({
  head: ["id", "name", "email", "createdAt", "updatedAt"],
  style: { head: ["cyan", "bold"] },
});
const user = result;
table.push([user.id, user.name, user.email, user.createdAt, user.updatedAt]);
console.log(table.toString());
"""

# Pide a Node que escriba con colores.
env = os.environ.copy()
env["FORCE_COLOR"] = "1"
env.pop("NO_COLOR", None)

# Lanza Node y muestra lo que escribe.
completed = subprocess.run(
    ["node", "--input-type=module", "-e", code],
    cwd=app,
    capture_output=True,
    text=True,
    env=env,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    print(completed.stderr, end="")
